In [ ]:
!pip install librosa speechrecognition textblob scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 50.0 MB/s eta 0:00:00


In [ ]:
from IPython.display import Javascript
from google.colab import output
from base64 import b64decode

def record_audio(filename='recorded_audio.wav', duration=5):
    js_code = f"""
    async function recordAudio() {{
        const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
        const recorder = new MediaRecorder(stream);
        const chunks = [];

        recorder.ondataavailable = e => chunks.push(e.data);

        recorder.start();
        await new Promise(resolve => setTimeout(resolve, {duration} * 1000));
        recorder.stop();

        await new Promise(resolve => recorder.onstop = resolve);

        const blob = new Blob(chunks);
        const arrayBuffer = await blob.arrayBuffer();
        const base64Audio = btoa(
            new Uint8Array(arrayBuffer)
                .reduce((data, byte) => data + String.fromCharCode(byte), '')
        );

        return base64Audio;
    }}
    recordAudio();
    """

    audio_data = output.eval_js(js_code)

    with open(filename, 'wb') as f:
        f.write(b64decode(audio_data))

    print(f"Audio saved as {filename}")


In [ ]:
record_audio('my_audio.wav', duration=8)


Audio saved as my_audio.wav


In [ ]:
from IPython.display import Audio
Audio('my_audio.wav')

In [ ]:
import librosa
file_name = 'my_audio.wav'
data, sample_rate = librosa.load(file_name)
print("Audio Loaded")


Audio Loaded


/tmp/ipykernel_339/3934615525.py:3: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sample_rate = librosa.load(file_name)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [ ]:
import numpy as np
mfcc = np.mean(librosa.feature.mfcc(y=data, sr=sample_rate),axis=1)
energy = np.mean(librosa.feature.rms(y=data))
print("MFCC", mfcc[:5])
print("Energy:",energy)

MFCC [-379.49927    90.34694     4.00816    31.049309    6.251367]
Energy: 0.034733165


In [ ]:
import speech_recognition as sr
import soundfile as sf

# The 'my_audio.wav' generated by the JS might not have proper WAV headers for speech_recognition.
# We will save the 'data' and 'sample_rate' (which librosa successfully loaded)
# into a new WAV file with correct headers.
output_filename = 'my_audio_standard.wav'
sf.write(output_filename, data, sample_rate)

r = sr.Recognizer()
with sr.AudioFile(output_filename) as source:
  # Corrected 'recognizer' to 'r'
  r.adjust_for_ambient_noise(source)
  audio_data = r.record(source)

text = r.recognize_google(audio_data)
print("Text:",text)


Text: I did not do anything trust me I'm telling the truth I swear I was not there trust me


In [ ]:
from textblob import TextBlob
analysis = TextBlob(text)
sentiment = analysis.sentiment.polarity
word_count = len(text.split())
print("Word Count:",word_count)
print("Sentiment:",sentiment)

Word Count: 19
Sentiment: 0.0


In [ ]:
features = np.hstack([mfcc, energy, sentiment, word_count])
print(features)

[-3.79499268e+02  9.03469391e+01  4.00816011e+00  3.10493088e+01
  6.25136709e+00 -1.30489101e+01 -1.98433077e+00 -1.99555206e+01
  4.20623016e+00  8.92343903e+00 -7.70127487e+00 -3.85821730e-01
 -8.07818353e-01 -2.13119626e+00 -4.59459066e+00 -5.47780371e+00
 -4.09276533e+00  1.65569973e+00 -3.82622743e+00 -3.37336540e+00
  3.47331651e-02  0.00000000e+00  1.90000000e+01]


In [ ]:
from sklearn.ensemble import RandomForestClassifier
RandomForestClassifier
X = [features, features*0.9,features*1.1]
y = [0, 1,0]
model = RandomForestClassifier()
model.fit(X,y)
pred = model.predict([features])
prob = model.predict_proba([features])
print("\n--- Result ---")
print("Text:", text)
print("Prediction:", "Lie" if pred[0] else "Truth")
print(f"Lie Probability: {prob[0][1]*100:.2f}%")
print(f"Truth Probability: {prob[0][0]*100:.2f}%")


--- Result ---
Text: I did not do anything trust me I'm telling the truth I swear I was not there trust me
Prediction: Truth
Lie Probability: 9.00%
Truth Probability: 91.00%
